<a href="https://colab.research.google.com/github/Vaquinhas/Service-Ticket-Data-Pipeline/blob/main/prj/Project-1/project_1_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Programming for AI

## Mini-project 1 — Service Ticket Data Pipeline

### Learning outcomes
- Turn data quality rules into a reusable NumPy/Pandas workflow.
- Preserve rejected data, transformation flags and an audit trail.
- Check edge cases and export reproducible results.

### Summary:

1. Diagnosis
2. Rejection Policy
3. Category Normalization
4. Numeric Treatment
5. Features
6. Reusable Function
7. Edge Cases
8. Export and Report

---

Read [BRIEF.md](BRIEF.md) before coding. Keep the synthetic input unchanged. Complete each task and restart/run from the beginning before submitting. **Contribution: 1.0/20 (5%).**

### Importing libraries and loading the data

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

candidates=[Path('service_tickets.csv'),Path('prj/Project-1/service_tickets.csv')]
source=next((p for p in candidates if p.is_file()),None)
if source is None: raise FileNotFoundError('Open from the project folder or upload service_tickets.csv to Colab')
raw=pd.read_csv(source)
print(raw.head())

  ticket_id   opened_on channel priority handling_minutes  satisfaction
0      T001  2026-10-01   email   medium             42.2           2.0
1      T002  2026-10-01   phone     high             76.3           3.0
2      T003  2026-10-01     web      low              NaN           4.0
3      T004  2026-10-01   email   medium            118.0           5.0
4      T005  2026-10-02   phone     high             99.5           1.0


---

## 1. Diagnosis

Report shape, schema, missingness, exact duplicates and repeated IDs. Explain the observation unit.

In [2]:
print("Dataset shape:", raw.shape)
print("Number of rows:", raw.shape[0])
print("Number of columns:", raw.shape[1])

print("\nDataset schema:")
raw.info()

print("\nMissing values per column:")
print(raw.isna().sum())

exact_duplicates = raw.duplicated().sum()
print("\nExact duplicate rows:", exact_duplicates)

repeated_ids = raw[raw.duplicated(subset=['ticket_id'], keep=False)]

print("\nRows with repeated ticket IDs:")
print(repeated_ids.sort_values('ticket_id'))

print("\nNumber of rows with repeated IDs:", len(repeated_ids))
print("Number of distinct repeated IDs:", repeated_ids['ticket_id'].nunique())

Dataset shape: (84, 6)
Number of rows: 84
Number of columns: 6

Dataset schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ticket_id         83 non-null     object 
 1   opened_on         84 non-null     object 
 2   channel           83 non-null     object 
 3   priority          83 non-null     object 
 4   handling_minutes  83 non-null     object 
 5   satisfaction      83 non-null     float64
dtypes: float64(1), object(5)
memory usage: 4.1+ KB

Missing values per column:
ticket_id           1
opened_on           0
channel             1
priority            1
handling_minutes    1
satisfaction        1
dtype: int64

Exact duplicate rows: 3

Rows with repeated ticket IDs:
   ticket_id   opened_on channel priority handling_minutes  satisfaction
0       T001  2026-10-01   email   medium             42.2           2.0
81      T001  

---

## 2. Rejection policy

Remove exact copies. Identify conflicting IDs and invalid IDs/dates. Preserve rejected source rows and all reasons.

In [3]:
deduplicated = raw.drop_duplicates().copy()

exact_duplicates_removed = len(raw) - len(deduplicated)

ticket_ids = deduplicated['ticket_id'].astype('string')

conflicting_id = (
    ticket_ids.notna() &
    ticket_ids.duplicated(keep=False)
)


missing_id = ticket_ids.isna()

malformed_id = (
    ticket_ids.notna() &
    ~ticket_ids.str.fullmatch(r'T\d{3}', na=False)
)


parsed_dates = pd.to_datetime(
    deduplicated['opened_on'],
    errors='coerce'
)

missing_date = deduplicated['opened_on'].isna()

unparseable_date = (
    deduplicated['opened_on'].notna() &
    parsed_dates.isna()
)

out_of_october = (
    parsed_dates.notna() &
    ~parsed_dates.between('2026-10-01', '2026-10-31')
)


checks = [
    (missing_id, 'missing_ticket_id'),
    (malformed_id, 'malformed_ticket_id'),
    (conflicting_id, 'conflicting_ticket_id'),
    (missing_date, 'missing_date'),
    (unparseable_date, 'unparseable_date'),
    (out_of_october, 'date_outside_october_2026')
]


rejection_reasons = []

for i in range(len(deduplicated)):
    row_reasons = [
        reason
        for condition, reason in checks
        if condition.iloc[i]
    ]

    rejection_reasons.append('; '.join(row_reasons))



rejection_mask = np.array([
    bool(reason) for reason in rejection_reasons
])


rejected_tickets = deduplicated.loc[rejection_mask].copy()

rejected_tickets['rejection_reason'] = [
    reason for reason in rejection_reasons if reason
]



eligible_tickets = deduplicated.loc[~rejection_mask].copy()


print("REJECTION POLICY SUMMARY")
print("------------------------")
print("Raw rows:", len(raw))
print("Exact duplicates removed:", exact_duplicates_removed)
print("Rejected rows:", len(rejected_tickets))
print("Eligible rows:", len(eligible_tickets))

print("\nCONFLICTING IDs")
print(deduplicated.loc[conflicting_id])

print("\nINVALID IDs")
print(deduplicated.loc[missing_id | malformed_id])

print("\nINVALID DATES")
print(deduplicated.loc[
    missing_date | unparseable_date | out_of_october
])

print("\nREJECTED TICKETS AND REASONS")
print(rejected_tickets)

REJECTION POLICY SUMMARY
------------------------
Raw rows: 84
Exact duplicates removed: 3
Rejected rows: 4
Eligible rows: 77

CONFLICTING IDs
   ticket_id   opened_on channel priority handling_minutes  satisfaction
39      T040  2026-10-10   email   medium             52.8           1.0
80      T040  2026-10-10   email   medium              999           1.0

INVALID IDs
   ticket_id   opened_on channel priority handling_minutes  satisfaction
34       NaN  2026-10-09   phone     high             45.4           1.0

INVALID DATES
   ticket_id   opened_on channel priority handling_minutes  satisfaction
33      T034  2026-10-32   email   medium              7.0           5.0

REJECTED TICKETS AND REASONS
   ticket_id   opened_on channel priority handling_minutes  satisfaction  \
33      T034  2026-10-32   email   medium              7.0           5.0   
34       NaN  2026-10-09   phone     high             45.4           1.0   
39      T040  2026-10-10   email   medium             52.8  

---

## 3. Category normalization

Normalize channel and priority and preserve unknown flags according to the brief.

In [4]:
clean_tickets = eligible_tickets.copy()

clean_tickets['channel'] = (
    clean_tickets['channel']
    .astype('string')
    .str.strip()
    .str.lower()
)

valid_channels = ['email', 'phone', 'web']

clean_tickets['channel_unknown'] = (
    ~clean_tickets['channel'].isin(valid_channels)
)

clean_tickets.loc[
    clean_tickets['channel_unknown'], 'channel'
] = 'unknown'


clean_tickets['priority'] = (
    clean_tickets['priority']
    .astype('string')
    .str.strip()
    .str.lower()
)

valid_priorities = ['low', 'medium', 'high']

clean_tickets['priority_unknown'] = (
    ~clean_tickets['priority'].isin(valid_priorities)
)

clean_tickets.loc[
    clean_tickets['priority_unknown'], 'priority'
] = 'unknown'


print("Channel distribution:")
print(clean_tickets['channel'].value_counts())

print("\nPriority distribution:")
print(clean_tickets['priority'].value_counts())

print("\nUnknown flags:")
print(clean_tickets[
    ['channel', 'channel_unknown', 'priority', 'priority_unknown']
].head(10))


Channel distribution:
channel
email      26
phone      25
web        24
unknown     2
Name: count, dtype: Int64

Priority distribution:
priority
high       26
low        25
medium     24
unknown     2
Name: count, dtype: Int64

Unknown flags:
  channel  channel_unknown priority  priority_unknown
0   email            False   medium             False
1   phone            False     high             False
2     web            False      low             False
3   email            False   medium             False
4   phone            False     high             False
5     web            False      low             False
6   email            False   medium             False
7   phone            False     high             False
8     web            False      low             False
9   email            False   medium             False


---

## 4. Numeric treatment

Treat durations and satisfaction differently. Preserve flags and explain why ratings must not be imputed.

In [ ]:
handling_numeric = pd.to_numeric(
    clean_tickets['handling_minutes'],
    errors='coerce'
)

valid_handling = (
    handling_numeric.notna() &
    np.isfinite(handling_numeric) &
    (handling_numeric >= 0)
)

clean_tickets['handling_imputed'] = ~valid_handling

handling_median = handling_numeric[valid_handling].median()

if pd.isna(handling_median):
    raise ValueError("No valid handling durations available for imputation.")

clean_tickets['handling_minutes'] = handling_numeric.fillna(
    handling_median
)

clean_tickets.loc[
    clean_tickets['handling_imputed'],
    'handling_minutes'
] = handling_median


---

## 5. Features

Compute safe duration scaling and the four required channel indicators; verify identifier alignment.

In [ ]:
# your code here

---

## 6. Reusable function

Consolidate your work into clean_tickets(raw), returning clean, rejected, features and audit. Do not mutate the input.

In [ ]:
# your code here

---

## 7. Edge cases

Test the two-ticket constant-duration example, conflicting IDs and the no-valid-duration error from the brief.

In [ ]:
# your code here

---

## 8. Export and report

Write required files to outputs/. Read the CSV files back, check row balance, and prepare REPORT.md.

In [ ]:
# your code here